In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df_gubernatorial = pd.read_csv("US_Election_2020/governors_county_candidate.csv")
df_presidential = pd.read_csv("US_Election_2020/president_county_candidate.csv")

display(df_gubernatorial.head())
display(df_presidential.head())

,state,county,candidate,party,votes,won
0,Delaware,Kent County,John Carney,DEM,44352,True
1,Delaware,Kent County,Julianne Murray,REP,39332,False
2,Delaware,Kent County,Kathy DeMatteis,IPD,1115,False
3,Delaware,Kent County,John Machurek,LIB,616,False
4,Delaware,New Castle County,John Carney,DEM,191678,True


,state,county,candidate,party,total_votes,won
0,Delaware,Kent County,Joe Biden,DEM,44552,True
1,Delaware,Kent County,Donald Trump,REP,41009,False
2,Delaware,Kent County,Jo Jorgensen,LIB,1044,False
3,Delaware,Kent County,Howie Hawkins,GRN,420,False
4,Delaware,New Castle County,Joe Biden,DEM,195034,True


In [4]:
# get the winners in each county and rename columns
winners_gov = df_gubernatorial[df_gubernatorial["won"] == True][["state", "county", "party", "votes"]].copy()
winners_gov = winners_gov.rename(columns={"party":"gov_party", "votes":"gov_votes"})

winners_pres = df_presidential[df_presidential["won"] == True][["state", "county", "party", "total_votes"]].copy()
winners_pres = winners_pres.rename(columns={"party":"pres_party", "total_votes":"pres_votes"})

# merge winners on state and county
df_winners = pd.merge(winners_pres, winners_gov, on=["state", "county"], how="inner")
df_winners.head()



,state,county,pres_party,pres_votes,gov_party,gov_votes
0,Delaware,Kent County,DEM,44552,DEM,44352
1,Delaware,New Castle County,DEM,195034,DEM,191678
2,Delaware,Sussex County,REP,71230,REP,68435
3,Indiana,Adams County,REP,10686,REP,9441
4,Indiana,Allen County,REP,92083,REP,98406


In [5]:
# data cleaning
df_winners = df_winners.dropna(subset=["state", "county"])
df_winners["pres_votes"] = pd.to_numeric(df_winners["pres_votes"], errors="coerce")
df_winners["gov_votes"]  = pd.to_numeric(df_winners["gov_votes"], errors="coerce")
df_winners = df_winners.dropna(subset=["pres_votes", "gov_votes"])

# final check
print("Merged winners shape (counties compared):", df_winners.shape)


Merged winners shape (counties compared): (1025, 6)


In [6]:
# create same-party indicator
df_winners["same_party_win"] = (df_winners["pres_party"] == df_winners["gov_party"])

# frequency: same party vs different
same_counts = df_winners["same_party_win"].value_counts(dropna=False)
print("Same party won both races (True/False):")
print(same_counts)
print("\nPercentage same party:", same_counts.get(True,0) / df_winners.shape[0] * 100)

# descriptive stats for vote totals (selective)
df_winners[["pres_votes", "gov_votes"]].describe()


Same party won both races (True/False):
same_party_win
True     720
False    305
Name: count, dtype: int64

Percentage same party: 70.24390243902438


,pres_votes,gov_votes
count,1025.000000,1025.000000
mean,12695.991220,12487.689756
std,42922.320095,42228.752855
min,5.000000,4.000000
25%,682.000000,742.000000
50%,2575.000000,2663.000000
75%,9573.000000,9076.000000
max,907310.000000,887374.000000
